# PoisonSpot × Narcissus — Facial Emotion Recognition (Kaggle)
**ResNet18 | CIS 582 | seed 42**

Full pipeline: trigger gen → poisoned training → batch-level provenance → sample-level provenance → scoring → retrain.

**Only one dataset required:** `sujaykapadnis/emotion-recognition-dataset` (attach before running).
All patched source files are embedded in this notebook.

The CONFIG cell below targets the **quick smoke-test** (5 epochs). Edit the full-run values before the final experiment run.

In [ ]:
# ════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════

import os as _os

# Auto-detect the emotion dataset class-folder root.
# Handles both flat  (dataset/angry/, dataset/happy/, …)
# and split layouts  (dataset/train/angry/, dataset/train/happy/, …).
_base = '/kaggle/input/emotion-recognition-dataset'
if not _os.path.exists(_base):
    # Fallback: scan /kaggle/input for any emotion-like dataset
    for _d in sorted(_os.listdir('/kaggle/input')):
        _candidate = _os.path.join('/kaggle/input', _d)
        if _os.path.isdir(_candidate):
            _base = _candidate
            break

def _find_emotion_cfg(base):
    """Walk base and return the first dir with >=4 subdirs containing images."""
    IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
    for root, dirs, _ in _os.walk(base):
        dirs.sort()
        if len(dirs) < 2:
            continue
        sub = _os.path.join(root, dirs[0])
        try:
            contents = _os.listdir(sub)
        except PermissionError:
            continue
        if any(_os.path.splitext(f)[1].lower() in IMG_EXTS for f in contents):
            return root
    return base

EMOTION_DIR  = _find_emotion_cfg(_base)
print('EMOTION_DIR auto-detected:', EMOTION_DIR)

REPO_DIR     = '/kaggle/working/PoisonSpot'
OUT_DIR      = '/kaggle/working/outputs'
SAVED_MODELS = f'{OUT_DIR}/saved_models'
PROV_PATH    = f'{OUT_DIR}/provenance'
RESULTS_PATH = f'{OUT_DIR}/results'
CFG_PATH     = f'{REPO_DIR}/configs/run_config_emotion.yaml'

# ── Attack / detection ───────────────────────────────────
# Class indices depend on alphabetical sort of emotion class folders:
# 0=angry 1=disgust 2=fear 3=happy 4=neutral 5=sad 6=surprise (typical)
TARGET_CLASS     = 3    # 'happy' by default (well-populated class)
PR_TGT           = 10  # % of target-class train images to poison
PR_SUS           = 50  # suspected-sample budget (%)
EPS              = 16  # L-inf trigger budget (0-255)
IMG_SIZE         = 112 # resize to this before feeding ResNet18

# ── Quick smoke test (5 epochs) ────────────────────────
# Swap for full-run values once smoke test passes.
SURROGATE_EPOCHS = 2   # full: 5
GEN_STEPS        = 50  # full: 200
EPOCHS           = 5   # full: 30
BATCH_SIZE       = 32  # full: 64
LR               = 0.01
GLOBAL_SEED      = 42
EP_BL_BASE       = 3   # full: 50
EP_BL            = 2   # full: 10
EP_SL_BASE       = 3   # full: 75
EP_SL            = 2   # full: 15
BS_BL            = 32  # full: 64
BS_SL            = 8   # full: 16
GROUPS           = 5   # full: 15
CV_MODEL         = 'RandomForest'
FORCE            = True  # full: False


In [ ]:
import subprocess, sys, os

try:
    r = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True)
    print('GPU:', r.stdout.strip() or 'NOT FOUND')
except FileNotFoundError:
    print('nvidia-smi not on PATH — trying torch')

import torch
print('CUDA:', torch.cuda.is_available(), '| PyTorch:', torch.__version__)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected')

# captum is required by scoring.py
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'pyyaml', 'tqdm', 'captum', '--no-deps'], check=True)
print('deps ok')


In [ ]:
import os, subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/Philenku/PoisonSpot', REPO_DIR],
        check=True)
    print('Cloned')
else:
    print('Repo already present')


In [ ]:
import os, base64
_dst = os.path.join(REPO_DIR, 'main.py')
os.makedirs(os.path.dirname(_dst), exist_ok=True)
_b64 = '...'
_content = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
with open(_dst, 'w', encoding='utf-8') as _f:
    _f.write(_content)
print('patched: main.py')


In [ ]:
import os, base64
_dst = os.path.join(REPO_DIR, 'src/helpers/data.py')
os.makedirs(os.path.dirname(_dst), exist_ok=True)
_b64 = '...'
_content = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
with open(_dst, 'w', encoding='utf-8') as _f:
    _f.write(_content)
print('patched: src/helpers/data.py')


In [ ]:
import os, base64
_dst = os.path.join(REPO_DIR, 'src/helpers/train.py')
os.makedirs(os.path.dirname(_dst), exist_ok=True)
_b64 = 'aW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgpmcm9tIHRxZG0gaW1wb3J0IHRxZG0KCgpkZWYgZXZhbHVhdGVfbW9kZWwobW9kZWwsIHRlc3RfbG9hZGVyLCBkZXZpY2UsICoqa3dhcmdzKToKICAgIG1vZGVsLmV2YWwoKQogICAgY29ycmVjdCwgdG90YWwgPSAwLCAwCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgYmF0Y2ggaW4gdGVzdF9sb2FkZXI6CiAgICAgICAgICAgIGlmIGlzaW5zdGFuY2UoYmF0Y2gsICh0dXBsZSwgbGlzdCkpIGFuZCBsZW4oYmF0Y2gpID49IDI6CiAgICAgICAgICAgICAgICB4LCB5ID0gYmF0Y2hbMF0udG8oZGV2aWNlKSwgYmF0Y2hbMV0udG8oZGV2aWNlKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcignVW5leHBlY3RlZCBiYXRjaDoge30nLmZvcm1hdCh0eXBlKGJhdGNoKSkpCiAgICAgICAgICAgIHByZWRzID0gbW9kZWwoeCkuYXJnbWF4KGRpbT0xKQogICAgICAgICAgICBjb3JyZWN0ICs9IChwcmVkcyA9PSB5KS5zdW0oKS5pdGVtKCkKICAgICAgICAgICAgdG90YWwgKz0geS5zaXplKDApCiAgICBhY2MgPSBjb3JyZWN0IC8gdG90YWwgaWYgdG90YWwgPiAwIGVsc2UgMC4wCiAgICByZXR1cm4gMC4wLCBhY2MKCgpkZWYgdHJhaW4obW9kZWwsIG9wdGltaXplciwgb3B0LCBzY2hlZHVsZXIsIGNyaXRlcmlvbiwKICAgICAgICAgIHRyYWluX2xvYWRlciwgdGVzdF9sb2FkZXIsIHBvaXNvbmVkX3Rlc3RfbG9hZGVyLAogICAgICAgICAgZXBvY2hzLCBnbG9iYWxfc2VlZD1Ob25lLCBkZXZpY2U9Tm9uZSwgY2ZnPU5vbmUsICoqa3dhcmdzKToKICAgIGlmIGRldmljZSBpcyBOb25lOgogICAgICAgIGRldmljZSA9IHRvcmNoLmRldmljZSgnY3VkYScgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKSBlbHNlICdjcHUnKQogICAgbW9kZWwgPSBtb2RlbC50byhkZXZpY2UpCiAgICBpZiBub3QgaXNpbnN0YW5jZShlcG9jaHMsIGludCkgb3IgZXBvY2hzIDw9IDA6CiAgICAgICAgZXBvY2hzID0gMQogICAgICAgIHByaW50KCdXQVJOSU5HOiBlcG9jaHMgZGVmYXVsdGVkIHRvIDEnKQoKICAgIHRyYWluX2NvcnJlY3QsIHRyYWluX3RvdGFsID0gMCwgMAoKICAgIGZvciBlcG9jaCBpbiByYW5nZShlcG9jaHMpOgogICAgICAgIG1vZGVsLnRyYWluKCkKICAgICAgICBwYmFyID0gdHFkbSh0cmFpbl9sb2FkZXIsIGRlc2M9J0Vwb2NoIHt9L3t9Jy5mb3JtYXQoZXBvY2ggKyAxLCBlcG9jaHMpKQogICAgICAgIGZvciBiYXRjaCBpbiBwYmFyOgogICAgICAgICAgICBpZiBpc2luc3RhbmNlKGJhdGNoLCAodHVwbGUsIGxpc3QpKSBhbmQgbGVuKGJhdGNoKSA+PSAyOgogICAgICAgICAgICAgICAgeCwgeSA9IGJhdGNoWzBdLnRvKGRldmljZSksIGJhdGNoWzFdLnRvKGRldmljZSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoJ1VuZXhwZWN0ZWQgYmF0Y2g6IHt9Jy5mb3JtYXQodHlwZShiYXRjaCkpKQogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKHNldF90b19ub25lPVRydWUpCiAgICAgICAgICAgIG91dHB1dHMgPSBtb2RlbCh4KQogICAgICAgICAgICBuX2NsYXNzZXMgPSBvdXRwdXRzLnNoYXBlWzFdCiAgICAgICAgICAgIHlfbWluLCB5X21heCA9IGludCh5Lm1pbigpLml0ZW0oKSksIGludCh5Lm1heCgpLml0ZW0oKSkKICAgICAgICAgICAgaWYgeV9taW4gPCAwIG9yIHlfbWF4ID49IG5fY2xhc3NlczoKICAgICAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoCiAgICAgICAgICAgICAgICAgICAgJ0xhYmVsIG91dCBvZiByYW5nZTogW3t9LCB7fV0gYnV0IG1vZGVsIGhhcyB7fSBvdXRwdXRzLicKICAgICAgICAgICAgICAgICAgICAuZm9ybWF0KHlfbWluLCB5X21heCwgbl9jbGFzc2VzKQogICAgICAgICAgICAgICAgKQogICAgICAgICAgICBsb3NzID0gY3JpdGVyaW9uKG91dHB1dHMsIHkpCiAgICAgICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICAgICAgICAgIHByZWRzID0gb3V0cHV0cy5hcmdtYXgoZGltPTEpCiAgICAgICAgICAgIHRyYWluX2NvcnJlY3QgKz0gKHByZWRzID09IHkpLnN1bSgpLml0ZW0oKQogICAgICAgICAgICB0cmFpbl90b3RhbCArPSB5LnNpemUoMCkKICAgICAgICAgICAgcGJhci5zZXRfcG9zdGZpeChsb3NzPSd7Oi40Zn0nLmZvcm1hdChsb3NzLml0ZW0oKSkpCgogICAgICAgIGlmIHNjaGVkdWxlciBpcyBub3QgTm9uZToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgc2NoZWR1bGVyLnN0ZXAoKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgICAgICBwcmludCgnV0FSTklORzogc2NoZWR1bGVyLnN0ZXAoKSBza2lwcGVkOiB7fScuZm9ybWF0KGUpKQoKICAgIF8sIHRlc3RfQUNDICAgPSBldmFsdWF0ZV9tb2RlbChtb2RlbCwgdGVzdF9sb2FkZXIsIGRldmljZSkKICAgIF8sIHRhcmdldF9BQ0MgPSBldmFsdWF0ZV9tb2RlbChtb2RlbCwgcG9pc29uZWRfdGVzdF9sb2FkZXIsIGRldmljZSkKICAgIHRyYWluX0FDQyAgPSB0cmFpbl9jb3JyZWN0IC8gdHJhaW5fdG90YWwgaWYgdHJhaW5fdG90YWwgPiAwIGVsc2UgMC4wCiAgICBjbGVhbl9BQ0MgID0gdGVzdF9BQ0MKCiAgICBwcmludCgnVHJhaW5pbmcgY29tcGxldGUgLS0gdHJhaW46IHs6LjRmfSB0ZXN0OiB7Oi40Zn0gYXNyOiB7Oi40Zn0nLmZvcm1hdCh0cmFpbl9BQ0MsIHRlc3RfQUNDLCB0YXJnZXRfQUNDKSkKICAgICMgUG9zaXRpb24gNCA9IEFTUiAodGFyZ2V0X0FDQyksIHNvICd0ZXN0X0FTUicgYXQgYWxsIGNhbGwgc2l0ZXMgZ2V0cyB0aGUgcmVhbCB2YWx1ZS4KICAgIHJldHVybiBtb2RlbCwgb3B0aW1pemVyLCBzY2hlZHVsZXIsIGZsb2F0KHRhcmdldF9BQ0MpLCBmbG9hdCh0ZXN0X0FDQyksIGZsb2F0KHRyYWluX0FDQyksIGZsb2F0KGNsZWFuX0FDQykK'
_content = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
with open(_dst, 'w', encoding='utf-8') as _f:
    _f.write(_content)
print('patched: src/helpers/train.py')


In [ ]:
import os, base64
_dst = os.path.join(REPO_DIR, 'src/helpers/provenance.py')
os.makedirs(os.path.dirname(_dst), exist_ok=True)
_b64 = '...'
_content = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
with open(_dst, 'w', encoding='utf-8') as _f:
    _f.write(_content)
print('patched: src/helpers/provenance.py')


In [ ]:
import os, base64
_dst = os.path.join(REPO_DIR, 'src/helpers/scoring.py')
os.makedirs(os.path.dirname(_dst), exist_ok=True)
_b64 = '...'
_content = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
with open(_dst, 'w', encoding='utf-8') as _f:
    _f.write(_content)
print('patched: src/helpers/scoring.py')


In [ ]:
import os, base64
_dst = os.path.join(REPO_DIR, 'src/__init__.py')
os.makedirs(os.path.dirname(_dst), exist_ok=True)
_b64 = 'IyBzcmMvX19pbml0X18ucHkgIChwYXRjaGVkIC0tIFNBL0hUL21peGVkL0xDIGltcG9ydHMgcmVtb3ZlZDsgbm90IG5lZWRlZCBmb3IgTmFyY2lzc3VzLUNhbHRlY2gyNTYpCgpmcm9tIC5hdHRhY2tzLk5hcmNpc3N1cy5nZW5lcmF0ZV9wb2lzb25fbmFyY2lzc3VzIGltcG9ydCAoCiAgICBnZXRfbmFyY2lzc3VzX2NpZmFyMTBfcG9pc29uZWRfZGF0YSwKKQoKIyBtb2RlbHMKZnJvbSAubW9kZWxzLnJlc25ldCBpbXBvcnQgUmVzTmV0CmZyb20gLm1vZGVscy5jdXN0b21fcmVzbmV0MTggaW1wb3J0IEN1c3RvbVJlc05ldDE4CmZyb20gLm1vZGVscy5kbm4gaW1wb3J0IEROTgpmcm9tIC5tb2RlbHMuY3VzdG9tX2NubiBpbXBvcnQgQ3VzdG9tQ05OCmZyb20gLm1vZGVscy5jdXN0b21fdml0IGltcG9ydCBDdXN0b21WaVQKCiMgdXRpbHMKZnJvbSAudXRpbHMudXRpbCBpbXBvcnQgQXZlcmFnZU1ldGVyCgojIEZJWCBJMTogYWRkZWQgZ2V0X25hcmNpc3N1c19jYWx0ZWNoMjU2X3BvaXNvbmVkX2RhdGEgdG8gdGhpcyBpbXBvcnQKZnJvbSAuaGVscGVycy5kYXRhIGltcG9ydCBnZXRfbG9hZGVyc19mcm9tX2RhdGFzZXQsIGdldF9yYW5kb21fcG9pc29uX2lkeCwgZ2V0X25hcmNpc3N1c19jYWx0ZWNoMjU2X3BvaXNvbmVkX2RhdGEsIGdldF9uYXJjaXNzdXNfZW1vdGlvbl9wb2lzb25lZF9kYXRhCmZyb20gLmhlbHBlcnMudHJhaW4gaW1wb3J0IHRyYWluLCBldmFsdWF0ZV9tb2RlbApmcm9tIC5oZWxwZXJzLnByb3ZlbmFuY2UgaW1wb3J0ICgKICAgIGNhcHR1cmVfZmlyc3RfbGV2ZWxfbXVsdGlfZXBvY2hfYmF0Y2hfc2FtcGxlX3dlaWdodF91cGRhdGVzLAogICAgY2FwdHVyZV9zYW1wbGVfbGV2ZWxfd2VpZ2h0X3VwZGF0ZXNfaWR2LAopCmZyb20gLmhlbHBlcnMuc2NvcmluZyBpbXBvcnQgdHJhaW5fcHJvdl9kYXRhX2N1c3RvbSwgc2NvcmVfcG9pc29uZWRfc2FtcGxlcywgZ2V0X2RpZmYKCl9fYWxsX18gPSBbCiAgICAjIGF0dGFja3MKICAgICJnZXRfbmFyY2lzc3VzX2NpZmFyMTBfcG9pc29uZWRfZGF0YSIsCiAgICAiZ2V0X25hcmNpc3N1c19jYWx0ZWNoMjU2X3BvaXNvbmVkX2RhdGEiLAogICAgImdldF9uYXJjaXNzdXNfZW1vdGlvbl9wb2lzb25lZF9kYXRhIiwKCiAgICAjIG1vZGVscwogICAgIlJlc05ldCIsCiAgICAiQ3VzdG9tUmVzTmV0MTgiLAogICAgIkN1c3RvbUNOTiIsCiAgICAiQ3VzdG9tVmlUIiwKICAgICJETk4iLAoKICAgICMgdXRpbHMKICAgICJBdmVyYWdlTWV0ZXIiLAoKICAgICMgZGF0YSBoZWxwZXJzCiAgICAiZ2V0X2xvYWRlcnNfZnJvbV9kYXRhc2V0IiwKICAgICJnZXRfcmFuZG9tX3BvaXNvbl9pZHgiLAoKICAgICMgdHJhaW5pbmcgaGVscGVycwogICAgInRyYWluIiwKICAgICJldmFsdWF0ZV9tb2RlbCIsCgogICAgIyBwcm92ZW5hbmNlIGNhcHR1cmUKICAgICJjYXB0dXJlX2ZpcnN0X2xldmVsX211bHRpX2Vwb2NoX2JhdGNoX3NhbXBsZV93ZWlnaHRfdXBkYXRlcyIsCiAgICAiY2FwdHVyZV9zYW1wbGVfbGV2ZWxfd2VpZ2h0X3VwZGF0ZXNfaWR2IiwKCiAgICAjIHNjb3JpbmcgLyBkZWZlbmNlCiAgICAidHJhaW5fcHJvdl9kYXRhX2N1c3RvbSIsCiAgICAic2NvcmVfcG9pc29uZWRfc2FtcGxlcyIsCiAgICAiZ2V0X2RpZmYiLApdCg=='
_content = base64.b64decode(_b64.encode('ascii')).decode('utf-8')
with open(_dst, 'w', encoding='utf-8') as _f:
    _f.write(_content)
print('patched: src/__init__.py')


In [ ]:
import subprocess
subprocess.run(
    ['find', REPO_DIR, '-name', '__pycache__', '-exec', 'rm', '-rf', '{}', '+'],
    capture_output=True)
print('pycache cleared')


In [ ]:
import os, yaml

for d in [SAVED_MODELS, PROV_PATH, RESULTS_PATH, os.path.dirname(CFG_PATH)]:
    os.makedirs(d, exist_ok=True)

cfg = {
    'exp':                  f'emotion_narcissus_eps{EPS}_tgt{PR_TGT}_seed{GLOBAL_SEED}',
    'attack':               'narcissus',
    'dataset':              'emotion',
    'model':                'ResNet18',
    'scenario':             'from_scratch',
    'dataset_dir':          EMOTION_DIR,
    'saved_models_path':    SAVED_MODELS + '/',
    'prov_path':            PROV_PATH + '/',
    'results_path':         RESULTS_PATH + '/',
    'clean_model_path':     '',
    'poisoned_model_path':  '',
    'retrained_model_path': '',
    'target_class':         TARGET_CLASS,
    'source_class':         TARGET_CLASS,
    'pr_tgt':               PR_TGT,
    'pr_sus':               PR_SUS,
    'eps':                  EPS,
    'img_size':             IMG_SIZE,
    'surrogate_epochs':     SURROGATE_EPOCHS,
    'gen_steps':            GEN_STEPS,
    'epochs':               EPOCHS,
    'bs':                   BATCH_SIZE,
    'lr':                   LR,
    'opt':                  'sgd',
    'global_seed':          GLOBAL_SEED,
    'gpu_id':               0,
    'clean_training':       False,
    'poisoned_training':    True,
    'batch_level':          True,
    'sample_level':         True,
    'score_samples':        True,
    'retrain':              True,
    'ep_bl':                EP_BL,
    'ep_bl_base':           EP_BL_BASE,
    'ep_sl':                EP_SL,
    'ep_sl_base':           EP_SL_BASE,
    'bs_bl':                BS_BL,
    'bs_sl':                BS_SL,
    'k_1':                  1,
    'k_2':                  0.0001,
    'groups':               GROUPS,
    'cv_model':             CV_MODEL,
    'threshold_type':       'Kmeans',
    'custom_threshold':     0.5,
    'vis':                  False,
    'get_result':           False,
    'force':                FORCE,
    'random':               False,
    'sample_from_test':     False,
}

with open(CFG_PATH, 'w') as fh:
    yaml.dump(cfg, fh, default_flow_style=False, sort_keys=False)
print(f'Config written: {CFG_PATH}')
print(f'epochs={EPOCHS}  ep_bl_base={EP_BL_BASE}  ep_sl_base={EP_SL_BASE}')


In [ ]:
import sys, os

# Verify dataset was found and has at least 2 class dirs
classes = [d for d in os.listdir(EMOTION_DIR)
           if os.path.isdir(os.path.join(EMOTION_DIR, d))]
print(f'Emotion dataset: {len(classes)} classes found')
print('  ->', sorted(classes))
assert len(classes) >= 2, f'Expected >=2 emotion classes, got {len(classes)}'
assert TARGET_CLASS < len(classes), \
    f'TARGET_CLASS={TARGET_CLASS} but only {len(classes)} classes present'

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

from src import (
    get_narcissus_emotion_poisoned_data,
    get_loaders_from_dataset, train, evaluate_model,
    capture_first_level_multi_epoch_batch_sample_weight_updates,
    capture_sample_level_weight_updates_idv,
    train_prov_data_custom, score_poisoned_samples, get_diff,
)
print('src imports OK')
print('Preflight passed — ready to run')


## Run pipeline

**Quick smoke test** (~15–30 min on T4): 5 epochs, confirms end-to-end on Kaggle.

Once confirmed, edit Cell 2 for the **full experiment run**:
```python
SURROGATE_EPOCHS = 5
GEN_STEPS        = 200
EPOCHS           = 30
BATCH_SIZE       = 64
EP_BL_BASE       = 25
EP_BL            = 5
EP_SL_BASE       = 25
EP_SL            = 5
BS_BL            = 64
BS_SL            = 16
GROUPS           = 10
FORCE            = False
```

**Target class note:** classes are sorted alphabetically by folder name. Default `TARGET_CLASS=3` targets `happy` (index 3 in angry/disgust/fear/happy/…). Verify with the preflight cell output and adjust if your dataset uses a different ordering.

In [ ]:
import subprocess, os, threading, time

env = os.environ.copy()
env["PYTHONPATH"]       = REPO_DIR
env["PYTHONIOENCODING"] = "utf-8"

proc = subprocess.Popen(
    ["python", "-u", "main.py", "-c", CFG_PATH],
    cwd=REPO_DIR, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)

def heartbeat():
    while proc.poll() is None:
        time.sleep(30)
        if proc.poll() is None:
            print(".", end="", flush=True)

t = threading.Thread(target=heartbeat, daemon=True)
t.start()

for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
print("
Exit code:", proc.returncode)
if proc.returncode != 0:
    raise RuntimeError("Pipeline failed -- check output above")


In [ ]:
import glob, os

print('=== Saved models ===')
for f in sorted(glob.glob(os.path.join(SAVED_MODELS, '*.pkl'))):
    print(' ', os.path.basename(f))

print('\n=== Results ===')
for f in sorted(glob.glob(os.path.join(RESULTS_PATH, '**'), recursive=True)):
    if os.path.isfile(f):
        print(' ', f.replace(RESULTS_PATH + '/', ''))

print('\n=== Provenance pkls ===')
for f in sorted(glob.glob(os.path.join(PROV_PATH, '*.pkl'))):
    print(' ', os.path.basename(f))
